<a href="https://colab.research.google.com/github/sun219361/Public-Data-Driven-Urban-Parking-Violation-Analysis/blob/main/ML_project_DININGCODE_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

다이닝 코드 음식장소 추천

In [ ]:
# ChromeDriver 설치
!apt-get update
!apt-get install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

!pip install webdriver_manager
!pip install --upgrade webdriver_manager
!pip install threadpoolctl==3.1.0

# Selenium 설치
!pip install selenium
!pip install --upgrade selenium


In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from urllib.request import urlopen
from selenium import webdriver
import re
import time
import urllib
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# Chrome 옵션 설정
chrome_options = Options()
chrome_options.add_argument('--headless')  # 브라우저를 표시하지 않고 실행할 경우

chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')


구별로 링크 가져오기

https://www.diningcode.com/?region=

https://www.diningcode.com/profile.php?rid=

In [ ]:
#서울시 구 정의 : 총 24개의 구
gu_list = ["강남구","강동구","강북구","강서구","관악구","광진구","구로구","금천구",
          "노원구","도봉구","동대문구","동작구","마포구","서대문구","서초구","성북구",
           "송파구","양천구","영등포구","용산구","은평구","종로구","중구", "중랑구"]


In [ ]:
# 다이닝 코드에서 식당 링크 가져오기
def get_links_DiningCode(gu):
    # Chrome 옵션 설정
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # 브라우저를 표시하지 않고 실행할 경우
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')

    # Chrome 드라이버 초기화
    driver = webdriver.Chrome(options=chrome_options)

    # 올바른 도시 코드 및 인코딩을 사용하여 URL 수정
    encoded_gu = urllib.parse.quote(gu)  # 한글 문자를 URL에 사용하기 위해 인코딩
    url = f"https://www.diningcode.com/list.php?query=서울%20{encoded_gu}"

    print("접근 중인 URL:", url)  # 이 줄을 추가하여 접근 중인 URL을 출력

    # 사용자 에이전트 헤더 설정
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
    }

    # HTTP GET 요청 보내기
    response = requests.get(url, headers=headers)

    # 로딩된 후 모든 링크 찾기
    link_elements = driver.find_elements(By.TAG_NAME, "a")
    res_links_raw = [link.get_attribute('href') for link in link_elements]

    res_links = [link for link in res_links_raw if 'profile' in link]
    res_links = list(set(res_links))
    Diningcode_links.extend(res_links)
    print(f'{gu}에서 {len(res_links)}곳 찾음')

    # 데이터가 충분하지 않은 경우
    if len(res_links) < 80:
        retry.append(gu)

    driver.close()

Diningcode_links = []
retry = []

for gu in gu_list:
    get_links_DiningCode(gu)

for gu in retry:
    get_links_DiningCode(gu)

len(Diningcode_links)


In [ ]:
Diningcode_links = list(set(Diningcode_links))

# save into text
with open("/content/drive/MyDrive/ML_data/restaurants_DiningCode.txt","w") as f:
    f.write('\n'.join(Diningcode_links))

크롤링
리뷰 & 가게

In [ ]:
D_restaurants_df = pd.DataFrame(columns=['name', 'address', 'category', 'main_mn', 'price', 'opng_tm', 'rating', 'rvw_cnt', 'tags'])
D_review_df = pd.DataFrame(columns=['res_name', 'user_name', 'rating', 'review'])

with open("/content/drive/MyDrive/ML_data/restaurants_DiningCode.txt", "r") as f:
    text = f.read()
urls = text.split()
urls

['https://www.diningcode.com/profile.php?rid=gymPpxy0wqyW',
 'https://www.diningcode.com/profile.php?rid=oFt7VQgJPD1x',
 'https://www.diningcode.com/profile.php?rid=rBGa94tjOhim',
 'https://www.diningcode.com/profile.php?rid=BIEVntjGgPBR',
 'https://www.diningcode.com/profile.php?rid=qYdn9yfYsqqR',
 'https://www.diningcode.com/profile.php?rid=ZbmKyMjf8LyV',
 'https://www.diningcode.com/profile.php?rid=X1677Q73zrTn',
 'https://www.diningcode.com/profile.php?rid=3m2pRph0pdjc',
 'https://www.diningcode.com/profile.php?rid=4pdt1C6gTjcU',
 'https://www.diningcode.com/profile.php?rid=xqY9zju9W4iZ',
 'https://www.diningcode.com/profile.php?rid=moKeujGlfTTW',
 'https://www.diningcode.com/profile.php?rid=GcNxur2JKeiv',
 'https://www.diningcode.com/profile.php?rid=WEh6gjz1tVd6',
 'https://www.diningcode.com/profile.php?rid=WIw4lV7nuN5O',
 'https://www.diningcode.com/profile.php?rid=9TUCHgdvAGHI',
 'https://www.diningcode.com/profile.php?rid=RPQbcSFyplWY',
 'https://www.diningcode.com/profile.php

In [ ]:
def crawling_reviews_DiningCode(name, link, driver):
    reviews = pd.DataFrame(columns=['res_name','user_name','rating','review'])
    try:
        driver.get(link)
    except:
        return None

    while True:
        try:
            click_more = driver.find_element_by_xpath("""//*[@id="div_more_review"]""")
            click_more.click()
        except:
            break

    time.sleep(1)

    try:
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        for user in soup.find('div',{'id':'div_review'}).find_all('div',{'id':re.compile('div_review_+')}):
            tmp = user.find('span','btxt').get_text()
            user_name = tmp[:tmp.find('(')].strip()
            user_rating = int(re.findall('\d+',user.find('i',{'style':re.compile('width:+')})['style'])[0])//20
            user_review = user.find('p','review_contents btxt').get_text().replace('\n','')

            reviews = reviews.append({'res_name':name,
                                    'user_name':user_name,
                                    'rating':user_rating,
                                    'review':user_review}, ignore_index=True)
    except:
        return None

    return reviews

In [ ]:
D_restaurants_df
D_review_df

,res_name,user_name,rating,review


In [ ]:
def crawling_res_data_DiningCode(link, driver):
    try:
        # Chrome 드라이버를 사용하여 링크 불러오기
        driver.get(link)

        # 페이지 로딩을 기다림 (예: 10초 동안 대기)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, 'tit'))
        )
    except Exception as e:
        print('링크 에러:', link)
        print(e)
        return None, None, None, None, None, None, None, None, None, None

    # 상호명
    try:
        name = driver.find_elements_by_class_name('tit')[3].text
        print(name)
    except:
        name = None

    # 주소
    try:
        address = driver.find_element_by_class_name('locat').text
    except:
        address = None

    # 업종(다이닝코드엔 업종이 없음)
    category = None

    # 주메뉴
    try:
        main_menu = driver.find_element_by_css_selector('div.menu-info.short p.l-txt').text.strip()
    except:
        main_menu = None

    # 가격(주메뉴)
    try:
        price = int(driver.find_element_by_css_selector('div.menu-info.short p.r-txt').text[:-1].replace(',', ''))
    except:
        price = None

    # 영업시간
    try:
        time_dict = dict()
        opening_time_list = driver.find_elements_by_css_selector('div.busi-hours.short p')[1:]
        for i in range(len(opening_time_list))[0::2]:
            if opening_time_list[i].text != '더보기':
                time_dict[opening_time_list[i].text] = opening_time_list[i+1].text

        opening_time = str(time_dict)
    except:
        opening_time = None

    # 평점과 리뷰 수
    try:
        ratings = float(driver.find_element_by_css_selector('strong.rate-point span').text)
        tmp = ratings[0].text.strip()
        rating = float(ratings[1].text[:-1])
        review_count = tmp[:tmp.find('명')]
    except:
        rating = None
        review_count = None

    # 태그
    try:
        tags = ','.join([i.text[:i.text.find('(')] for i in driver.find_elements_by_css_selector('p.icon')])
    except:
        tags = None

    # 리뷰 데이터 가져오기
    res_reviews = crawling_reviews_DiningCode(name, link, driver)

    return name, address, category, main_menu, price, opening_time, rating, review_count, tags, res_reviews


In [ ]:
count = 1
for url in urls:
    try:

        chrome_path = '/usr/lib/chromium-browser/chromedriver.exe'
        driver = webdriver.Chrome(options=chrome_options)

        print(count, '\t|', end='')

        name, address, category, main_menu, price, opening_time, rating, review_count, tags, review_df = crawling_res_data_DiningCode(url, driver)

        D_restaurants_df = D_restaurants_df.append({
            'name': name,
            'address': address,
            'category': category,
            'main_mn': main_menu,
            'price': price,
            'opng_tm': opening_time,
            'rating': rating,
            'rvw_cnt': review_count,
            'tags': tags
        }, ignore_index=True)

        D_review_df = D_review_df.append(review_df, ignore_index=True)

        count += 1
    finally:
        if 'driver' in locals():
            driver.quit()




In [ ]:
D_restaurants_df.to_csv('/content/drive/MyDrive/ML_data/DiningCode_df.csv', index=False)
D_review_df.to_csv('/content/drive/MyDrive/ML_data/DiningCode_review_df.csv', index=False)


LabelEncoder를 사용하여 식당 이름과 사용자 이름을 숫자 값으로 변환

문장부호 제거

번역 후 라벨 추가

In [ ]:
!pip install pypapago

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm
from pypapago import Translator

In [ ]:
# Load DiningCode data
eat_info = pd.read_csv("/content/drive/MyDrive/ML_data/DiningCode_df.csv")
eat_review = pd.read_csv("/content/drive/MyDrive/ML_data/DiningCode_review_df.csv")

In [ ]:
eat_info.head()

,name,address,category,main_mn,price,opng_tm,rating,rvw_cnt,tags
0,놀부만두,서울특별시 동대문구 휘경2동 276-33,NaN,냉모밀,6000.0,"{'월-금,일': '오전 11시 - 오후 8시 30분 '}",NaN,NaN,"점심식사,혼밥,저녁식사,간식,지역주민이 찾는,서민적인,시끌벅적한,가성비좋은,푸짐한,..."
1,호반,서울특별시 종로구 낙원동 85-7,NaN,순대국,8000.0,{'매일': '오전 11시 30분 - 오후 10시 (준비시간 15:00~16:30)...,NaN,NaN,"저녁식사,점심식사,식사모임,가족외식,술모임,접대,혼술,데이트,혼밥,회식,아침식사,서..."
2,민벅 건대점,서울특별시 광진구 화양동 10-1,NaN,알리오올리오,9900.0,{'매일': '오전 11시 30분 - 오후 10시 (구정 추석 당일 3시 오픈)'},NaN,NaN,"점심식사,저녁식사,데이트,여자끼리,가족외식,모임,기념일,식사모임,아침식사,이국적/이..."
3,뉴델리,서울시 동대문구 회기동 3-182,NaN,치킨 탄두리 (한마리),15000.0,{'매일': '오전 11시 - 오후 10시 '},NaN,NaN,"점심식사,저녁식사,데이트,식사모임,이국적/이색적,깔끔한,캐주얼한"
4,강릉스낵,서울특별시 양천구 목동 406-65,NaN,직송 모듬 회,60000.0,"{'월-금': '오후 5시 - 오전 1시 ', '토/일': '오후 4시 - 오전 1...",NaN,NaN,"술모임,점심식사,저녁식사,간식,데이트,아이동반,가족외식,식사모임,혼밥,서민적인,캐주..."


In [ ]:
eat_review.head()

,res_name,user_name,rating,review
0,놀부만두,Youngmi Lee,5,지역맛집답게 적당히 허름하고 작지만 손님이 꽉차 있었다. 냉모밀 국물은 깊은달달함~...
1,놀부만두,Kunwoo Kim,4,제기동 평양냉면 갔다가 정기휴일 발길돌려휘경동 생방송투데이 맛집 놀부만두로 아버지와...
2,놀부만두,erica,3,비빔모밀 온모밀 김치만두 먹어봄유명세에 비해 그닥...그나마 셋 중에는 김치만두가 ...
3,놀부만두,발챙이,4,김치만두랑 냉모밀 조합이 최고입니다. 만두피가 엄청 쫄깃해요.
4,놀부만두,nelia,5,NaN


In [ ]:
# Drop NaN values
eat_info = eat_info[eat_info['name'].notnull()]
eat_review = eat_review[eat_review['user_name'].notnull()]

# Reset index
eat_info.reset_index(drop=True, inplace=True)
eat_review.reset_index(drop=True, inplace=True)

# Delete overlapped data
# 식당 정보 데이터 세트에서 중복 제거. 중복은 식당 이름 기준
for id in eat_info.index:
    if list(eat_info['name']).count(eat_info.loc[id, 'name']) > 1:
        eat_info.drop(id, inplace=True)


In [ ]:
# Label encoding, Place_name -> p_id, User_name -> u_id
# LabelEncoder를 사용하여 식당 이름과 사용자 이름을 숫자 값으로 변환
# (p_id는 place_id 및 u_id는 user_id로) 인코딩
le = LabelEncoder()
le.fit(eat_info['name'])
eat_info['p_id'] = le.transform(eat_info['name'])
eat_review['p_id'] = le.transform(eat_review['res_name'])

user_le = LabelEncoder()
user_le.fit(eat_review['user_name'])
eat_review['u_id'] = user_le.transform(eat_review['user_name'])



In [ ]:
# Delete punctuation marks
# 리뷰에서 문장 부호 제거
def delete_punctuation_marks_review(review):
    review = review.replace('.', '. ')
    review = review.replace('!', '! ')
    review = review.replace('(', '( ')
    review = review.replace(')', ') ')
    review = review.replace('^', '')
    review = review.replace('*', '')
    review = review.replace('-', ' ')
    review = review.replace('\n', '. ')
    review = review.replace('\"', '. ')
    return review


In [ ]:
eat_info.head()

,name,address,category,main_mn,price,opng_tm,rating,rvw_cnt,tags,p_id
0,놀부만두,서울특별시 동대문구 휘경2동 276-33,NaN,냉모밀,6000.0,"{'월-금,일': '오전 11시 - 오후 8시 30분 '}",NaN,NaN,"점심식사,혼밥,저녁식사,간식,지역주민이 찾는,서민적인,시끌벅적한,가성비좋은,푸짐한,...",304
1,호반,서울특별시 종로구 낙원동 85-7,NaN,순대국,8000.0,{'매일': '오전 11시 30분 - 오후 10시 (준비시간 15:00~16:30)...,NaN,NaN,"저녁식사,점심식사,식사모임,가족외식,술모임,접대,혼술,데이트,혼밥,회식,아침식사,서...",2234
2,민벅 건대점,서울특별시 광진구 화양동 10-1,NaN,알리오올리오,9900.0,{'매일': '오전 11시 30분 - 오후 10시 (구정 추석 당일 3시 오픈)'},NaN,NaN,"점심식사,저녁식사,데이트,여자끼리,가족외식,모임,기념일,식사모임,아침식사,이국적/이...",770
3,뉴델리,서울시 동대문구 회기동 3-182,NaN,치킨 탄두리 (한마리),15000.0,{'매일': '오전 11시 - 오후 10시 '},NaN,NaN,"점심식사,저녁식사,데이트,식사모임,이국적/이색적,깔끔한,캐주얼한",311
4,강릉스낵,서울특별시 양천구 목동 406-65,NaN,직송 모듬 회,60000.0,"{'월-금': '오후 5시 - 오전 1시 ', '토/일': '오후 4시 - 오전 1...",NaN,NaN,"술모임,점심식사,저녁식사,간식,데이트,아이동반,가족외식,식사모임,혼밥,서민적인,캐주...",81


In [ ]:
eat_review.head()

,res_name,user_name,rating,review,p_id,u_id
0,놀부만두,Youngmi Lee,5,지역맛집답게 적당히 허름하고 작지만 손님이 꽉차 있었다. 냉모밀 국물은 깊은달달함~...,304,1616
1,놀부만두,Kunwoo Kim,4,제기동 평양냉면 갔다가 정기휴일 발길돌려휘경동 생방송투데이 맛집 놀부만두로 아버지와...,304,905
2,놀부만두,erica,3,비빔모밀 온모밀 김치만두 먹어봄유명세에 비해 그닥...그나마 셋 중에는 김치만두가 ...,304,1817
3,놀부만두,발챙이,4,김치만두랑 냉모밀 조합이 최고입니다. 만두피가 엄청 쫄깃해요.,304,3911
4,놀부만두,nelia,5,NaN,304,2151


In [ ]:
eat_info['reviews'] = ''

In [ ]:
ts = Translator()
eat_review['review_eng'] = ''

In [ ]:
eat_info.head()

,name,address,category,main_mn,price,opng_tm,rating,rvw_cnt,tags,p_id,reviews
0,놀부만두,서울특별시 동대문구 휘경2동 276-33,NaN,냉모밀,6000.0,"{'월-금,일': '오전 11시 - 오후 8시 30분 '}",NaN,NaN,"점심식사,혼밥,저녁식사,간식,지역주민이 찾는,서민적인,시끌벅적한,가성비좋은,푸짐한,...",304,
1,호반,서울특별시 종로구 낙원동 85-7,NaN,순대국,8000.0,{'매일': '오전 11시 30분 - 오후 10시 (준비시간 15:00~16:30)...,NaN,NaN,"저녁식사,점심식사,식사모임,가족외식,술모임,접대,혼술,데이트,혼밥,회식,아침식사,서...",2234,
2,민벅 건대점,서울특별시 광진구 화양동 10-1,NaN,알리오올리오,9900.0,{'매일': '오전 11시 30분 - 오후 10시 (구정 추석 당일 3시 오픈)'},NaN,NaN,"점심식사,저녁식사,데이트,여자끼리,가족외식,모임,기념일,식사모임,아침식사,이국적/이...",770,
3,뉴델리,서울시 동대문구 회기동 3-182,NaN,치킨 탄두리 (한마리),15000.0,{'매일': '오전 11시 - 오후 10시 '},NaN,NaN,"점심식사,저녁식사,데이트,식사모임,이국적/이색적,깔끔한,캐주얼한",311,
4,강릉스낵,서울특별시 양천구 목동 406-65,NaN,직송 모듬 회,60000.0,"{'월-금': '오후 5시 - 오전 1시 ', '토/일': '오후 4시 - 오전 1...",NaN,NaN,"술모임,점심식사,저녁식사,간식,데이트,아이동반,가족외식,식사모임,혼밥,서민적인,캐주...",81,


In [ ]:
eat_review.head()

,res_name,user_name,rating,review,p_id,u_id,review_eng
0,놀부만두,Youngmi Lee,5,지역맛집답게 적당히 허름하고 작지만 손님이 꽉차 있었다. 냉모밀 국물은 깊은달달함~...,304,1616,
1,놀부만두,Kunwoo Kim,4,제기동 평양냉면 갔다가 정기휴일 발길돌려휘경동 생방송투데이 맛집 놀부만두로 아버지와...,304,905,
2,놀부만두,erica,3,비빔모밀 온모밀 김치만두 먹어봄유명세에 비해 그닥...그나마 셋 중에는 김치만두가 ...,304,1817,
3,놀부만두,발챙이,4,김치만두랑 냉모밀 조합이 최고입니다. 만두피가 엄청 쫄깃해요.,304,3911,
4,놀부만두,nelia,5,NaN,304,2151,


In [ ]:
!pip uninstall googletrans

In [ ]:
!pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 7.0 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17396 sha256=e0e66b0f5175f5a2c9b98276c11519d3a98d2b63642cb5c91d5d8c6b2cbe202a
  Stored in directory: /root/.cache/pip/wheels/c0/59/9f/7372f0cf70160fe61b528532e1a7c8498c4becd6bcffb022de
Successfully built googletrans
  Attempting uninstall: chardet
    Found existing installation: chardet 5.2.0
    Uninstalling c

In [ ]:
from tqdm import tqdm
from googletrans import Translator
import pandas as pd

# 리뷰 번역 함수
def translate_review(review):
    try:
        translator = Translator()
        translated_review = translator.translate(review, src='ko', dest='en').text
        return translated_review
    except Exception as e:
        print(f"번역 오류: {e}")
        return ''

# 리뷰 번역 및 결과 출력
for id in tqdm(eat_review.index):
    review = eat_review.loc[id, 'review']

    if pd.isnull(review):
        continue

    # 리뷰가 이미 번역되었는지 확인
    if pd.isnull(eat_review.loc[id, 'review_eng']):
        review_pre = delete_punctuation_marks_review(review)
        review_ts = translate_review(review_pre)

        eat_review.loc[id, 'review_eng'] = review_ts
        print(f"원본: {review}, 번역: {review_ts}")

In [ ]:
# 각 레스토랑에 대한 번역된 리뷰 수집
for id in eat_info['p_id']:
    review_all = ''

    for review in eat_review[eat_review['p_id'] == id]['review_eng']:
        if isinstance(review, str):
            review_all += review + '. '
        elif isinstance(review, float):
            review_all += str(review) + '. '

    eat_info.loc[eat_info['p_id'] == id, 'reviews'] = review_all


In [ ]:
eat_info.head()

,name,address,category,main_mn,price,opng_tm,rating,rvw_cnt,tags,p_id,reviews
0,놀부만두,서울특별시 동대문구 휘경2동 276-33,NaN,냉모밀,6000.0,"{'월-금,일': '오전 11시 - 오후 8시 30분 '}",NaN,NaN,"점심식사,혼밥,저녁식사,간식,지역주민이 찾는,서민적인,시끌벅적한,가성비좋은,푸짐한,...",304,"Like a local restaurant, it was moderately sha..."
1,호반,서울특별시 종로구 낙원동 85-7,NaN,순대국,8000.0,{'매일': '오전 11시 30분 - 오후 10시 (준비시간 15:00~16:30)...,NaN,NaN,"저녁식사,점심식사,식사모임,가족외식,술모임,접대,혼술,데이트,혼밥,회식,아침식사,서...",2234,It is a Korean restaurant located at the edge ...
2,민벅 건대점,서울특별시 광진구 화양동 10-1,NaN,알리오올리오,9900.0,{'매일': '오전 11시 30분 - 오후 10시 (구정 추석 당일 3시 오픈)'},NaN,NaN,"점심식사,저녁식사,데이트,여자끼리,가족외식,모임,기념일,식사모임,아침식사,이국적/이...",770,"If you have 1 menu per person, it will give yo..."
3,뉴델리,서울시 동대문구 회기동 3-182,NaN,치킨 탄두리 (한마리),15000.0,{'매일': '오전 11시 - 오후 10시 '},NaN,NaN,"점심식사,저녁식사,데이트,식사모임,이국적/이색적,깔끔한,캐주얼한",311,You can enjoy delicious Indian curry at an old...
4,강릉스낵,서울특별시 양천구 목동 406-65,NaN,직송 모듬 회,60000.0,"{'월-금': '오후 5시 - 오전 1시 ', '토/일': '오후 4시 - 오전 1...",NaN,NaN,"술모임,점심식사,저녁식사,간식,데이트,아이동반,가족외식,식사모임,혼밥,서민적인,캐주...",81,"Tteokbokki is spicy, so it's delicious.The fri..."


In [ ]:
eat_review.head()

,res_name,user_name,rating,review,p_id,u_id,review_eng
0,놀부만두,Youngmi Lee,5,지역맛집답게 적당히 허름하고 작지만 손님이 꽉차 있었다. 냉모밀 국물은 깊은달달함~...,304,1616,"Like a local restaurant, it was moderately sha..."
1,놀부만두,Kunwoo Kim,4,제기동 평양냉면 갔다가 정기휴일 발길돌려휘경동 생방송투데이 맛집 놀부만두로 아버지와...,304,905,I went to Pyongyang cold noodles and turned on...
2,놀부만두,erica,3,비빔모밀 온모밀 김치만두 먹어봄유명세에 비해 그닥...그나마 셋 중에는 김치만두가 ...,304,1817,Bibim Mumil Onmemin Kimchi Band only eat bumen...
3,놀부만두,발챙이,4,김치만두랑 냉모밀 조합이 최고입니다. 만두피가 엄청 쫄깃해요.,304,3911,Kimchi -man dumplings and cold hair combinatio...
4,놀부만두,nelia,5,NaN,304,2151,NaN


In [ ]:
# Save to CSV
eat_info.to_csv('/content/drive/MyDrive/ML_data/translated_eat_info.csv', index=False)
eat_review.to_csv('/content/drive/MyDrive/ML_data/translated_eat_review.csv', index=False)

prevent unnecessary API calls for already translated reviews

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm

from tqdm import tqdm
from googletrans import Translator

# DiningCode 데이터 로드
eat_info = pd.read_csv("/content/drive/MyDrive/ML_data/translated_eat_info.csv")
eat_review = pd.read_csv("/content/drive/MyDrive/ML_data/translated_eat_review.csv")

# 리뷰 번역 함수
def translate_review(review):
    try:
        translator = Translator()
        translated_review = translator.translate(review, src='ko', dest='en').text
        return translated_review
    except Exception as e:
        print(f"번역 오류: {e}")
        return ''

# 리뷰 번역 및 결과 출력
for id in tqdm(eat_review.index):
    review = eat_review.loc[id, 'review']

    if pd.isnull(review):
        continue

    # 리뷰가 이미 번역되었는지 확인
    if pd.isnull(eat_review.loc[id, 'review_eng']):
        review_pre = delete_punctuation_marks_review(review)
        review_ts = translate_review(review_pre)

        eat_review.loc[id, 'review_eng'] = review_ts
        print(f"원본: {review}, 번역: {review_ts}")


 32%|███▏      | 7091/22421 [00:00<00:01, 15321.49it/s]

번역 오류: the JSON object must be str, bytes or bytearray, not NoneType
원본: ^^, 번역: 


 80%|████████  | 17976/22421 [00:00<00:00, 24608.24it/s]

원본: 소문듣고 점심시간 지나 15시 도착 웨이팅 없이 한가하게 식사 칼국수 동네에서 먹던 멸치국물이 아니라 뭔가 진한게 넘어가는 식감 좋았으며 면도 무척 부드러웠음 만두도 속이 꽉차 육즙이 있어 좋았음, 번역: After listening to rumors, the arrival time after lunchtime was 15 o'clock without a weight, but it was not anchovy broth that was eaten in the kalguksu neighborhood, but something rich and the texture was good and the shave was very soft.
원본: 명동하면 빠질 수 없는 맛집 명동교자본점!미쉐린 가이드 2020에 선정된 만큼 외국인들도 굉장히 많이 오는 칼국수집인데요.칼국수 + 비빔국수 + 만두를 주문해봤습니다😄달달한 칼국수랑 매콤한 비빔국수에 사이드 만두까지 조합이 아주 좋아요👍👍 추천추천, 번역: Myeongdong Kyo Country Store that you can't miss if you do not miss Myeongdong!As it was selected for the Michelin Guide 2020, it is a kalguksu restaurant where many foreigners come.I ordered kalguksu + bibim noodles + dumplings 😄 Sweet kalguksu and spicy bibim noodles and side dumplings are very good.
원본: 전체적으로 칼국수와 만두의 고기가 굉장히 기름집니다 고기 육수 기름진 맛 좋아하시면 추천합니다 김치가 마늘 양념이 많아서 국물에 타 먹기 좋습니다 전 콩국수가 정말 맛있었어요 국물이 진하고 정말 고소합니다, 번역: Overall, the meat of kalguksu and dumplings is very oily.
원본: 명동교

 88%|████████▊ | 19641/22421 [00:21<00:07, 364.27it/s]  

...

번역 오류: 'Translator' object has no attribute 'raise_Exception'
원본: 깨끗하고좋음 맛도 그럭저럭있음가성비가 좋은편은 아닌듯 대신양많고 셀프포장대 있음, 번역: 


100%|█████████▉| 22416/22421 [13:12<00:02,  1.99it/s]

번역 오류: 'Translator' object has no attribute 'raise_Exception'
원본: 돈까스 고기가 많이 얇고 튀김옷맛이 강합니다.다른 부분은 soso 합니다., 번역: 


100%|██████████| 22421/22421 [13:12<00:00, 28.27it/s]

번역 오류: 'Translator' object has no attribute 'raise_Exception'
원본: 돈까스 품질은 중상.  세트메뉴의 국물우동과 볶음우동의 퀄러티는 중하. 돈까스만 먹는걸 추천합니다., 번역: 


In [ ]:
# 각 레스토랑에 대한 번역된 리뷰 수집
for id in eat_info['p_id']:
    review_all = ''

    for review in eat_review[eat_review['p_id'] == id]['review_eng']:
        if isinstance(review, str):
            review_all += review + '. '
        elif isinstance(review, float):
            review_all += str(review) + '. '

    eat_info.loc[eat_info['p_id'] == id, 'reviews'] = review_all


In [ ]:
# CSV로 저장
eat_info.to_csv('/content/drive/MyDrive/ML_data/translated_eat_info.csv', index=False)
eat_review.to_csv('/content/drive/MyDrive/ML_data/translated_eat_review.csv', index=False)

음식 정보와 사용자 리뷰 데이터

태그 기반으로 KMeans 클러스터링

->  res_matrix와 user_matrix 데이터프레임을 생성

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from gensim import models
from gensim.models import KeyedVectors
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm

from collections import Counter

In [ ]:
def delete_punctuation_marks_tag(word):
    word = word.replace(",","")
    word = word.replace(".","")
    word = word.replace(",","")
    word = word.replace("!","")
    word = word.replace(":","")
    word = word.replace("?","")
    word = word.replace("~","")
    word = word.replace("\\","")
    word = word.replace("\"","")
    word = word.replace(";","")
    return word.lower()

In [ ]:
eat_info = pd.read_csv('/content/drive/MyDrive/ML_data/translated_eat_info.csv')
eat_review = pd.read_csv('/content/drive/MyDrive/ML_data/translated_eat_review.csv')

In [ ]:
eat_review_by_user = pd.DataFrame(index = eat_review['u_id'].unique(), columns=['reviews'])

for id in tqdm(eat_review['u_id'].unique()):
    review_all = ''
    for review in eat_review.loc[eat_review['u_id']==id,'review_eng']:
        if type(review) != float:
            review_all += review+'. '
    eat_review_by_user.loc[id,'reviews'] = review_all


  0%|          | 0/6181 [00:00<?, ?it/s]

In [ ]:
eat_info['tags']=''
eat_review_by_user['tags']=''

In [ ]:
eat_info.head()

,name,address,category,main_mn,price,opng_tm,rating,rvw_cnt,tags,p_id,reviews
0,놀부만두,서울특별시 동대문구 휘경2동 276-33,NaN,냉모밀,6000.0,"{'월-금,일': '오전 11시 - 오후 8시 30분 '}",NaN,NaN,,304,"Like a local restaurant, it was moderately sha..."
1,호반,서울특별시 종로구 낙원동 85-7,NaN,순대국,8000.0,{'매일': '오전 11시 30분 - 오후 10시 (준비시간 15:00~16:30)...,NaN,NaN,,2234,It is a Korean restaurant located at the edge ...
2,민벅 건대점,서울특별시 광진구 화양동 10-1,NaN,알리오올리오,9900.0,{'매일': '오전 11시 30분 - 오후 10시 (구정 추석 당일 3시 오픈)'},NaN,NaN,,770,"If you have 1 menu per person, it will give yo..."
3,뉴델리,서울시 동대문구 회기동 3-182,NaN,치킨 탄두리 (한마리),15000.0,{'매일': '오전 11시 - 오후 10시 '},NaN,NaN,,311,You can enjoy delicious Indian curry at an old...
4,강릉스낵,서울특별시 양천구 목동 406-65,NaN,직송 모듬 회,60000.0,"{'월-금': '오후 5시 - 오전 1시 ', '토/일': '오후 4시 - 오전 1...",NaN,NaN,,81,"Tteokbokki is spicy, so it's delicious.The fri..."


In [ ]:
eat_review_by_user.head()

,reviews,tags
1616,"Like a local restaurant, it was moderately sha...",
905,I went to Pyongyang cold noodles and turned on...,
1817,Bibim Mumil Onmemin Kimchi Band only eat bumen...,
3911,Kimchi -man dumplings and cold hair combinatio...,
2151,,


In [ ]:
with open('/content/drive/MyDrive/ML_data/stop_words.txt','r',-1,'utf-8') as f:
    stop_words = set(f.read().split('/'))

stop_words

{'',
 '*',
 '+',
 '+0',
 '+_+',
 '0th',
 '1',
 '1st',
 '2',
 '2nd',
 '3',
 '3rd',
 '4',
 '4b',
 '4th',
 '5th',
 '6',
 '6th',
 '7th',
 '8th',
 '9',
 '=',
 '>',
 '@',
 '__',
 '___________________________________',
 'a',
 'a4',
 'about',
 'above',
 'accumulated',
 'acrostic',
 'ade',
 'admirable',
 'adzuki',
 'after',
 'again',
 'against',
 'ago',
 'agre',
 'ah',
 'ahead',
 'ai',
 'ain',
 'ajumma',
 'aka',
 'al',
 'albert',
 'all',
 'alla',
 'alle',
 'alli',
 'allthe',
 'alma',
 'alongthe',
 'altro',
 'am',
 'amazing',
 'ame',
 'americano',
 'amo',
 'an',
 'anam',
 'ancestral',
 'ancient',
 'and',
 'anniversary',
 'annual',
 'ant',
 'anticipated',
 'any',
 'apo',
 'arabian',
 'archa',
 'are',
 'aren',
 "aren't",
 'argent',
 'arti',
 'as',
 'asu',
 'at',
 'attempted',
 'austrian',
 'avai',
 'average',
 'avo',
 'awesome',
 'b',
 'baby',
 'badam',
 'badi',
 'bahn',
 'bal',
 'baltic',
 'bam',
 'bb',
 'be',
 'bean',
 'beans',
 'because',
 'been',
 'beep',
 'bef',
 'before',
 'beforethe',
 'beg

In [ ]:
def get_tags_to_txt(text):
    if type(text) != str:
        return ''
    text = word_tokenize(text)
    tags_dump = nltk.pos_tag(text)
    tags = []

    for word in tags_dump:
        if (word[1] == 'JJ') and (word[0] not in stop_words):
            tags.append(delete_punctuation_marks_tag(word[0]))

    tags = ','.join(tags)

    return tags


In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger', quiet=True)


True

In [ ]:
for i in tqdm(eat_info.index):
    eat_info.loc[i,'tags'] = get_tags_to_txt(eat_info.loc[i,'reviews'])

for i in tqdm(eat_review_by_user.index):
    eat_review_by_user.loc[i,'tags'] = get_tags_to_txt(eat_review_by_user.loc[i,'reviews'])

  0%|          | 0/2314 [00:00<?, ?it/s]

  0%|          | 0/6181 [00:00<?, ?it/s]

word2vec model :

https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?resourcekey=0-wjGZdNAUop6WykTtMip30g

Word2Vec을 사용하여 앞선 tokenizing 단계에서 추출해낸 형용사를 300차원의 벡터로 변형한 후,

K-means 알고리즘을 사용해 클러스터링 작업을 수행

In [ ]:
import gensim
from gensim.models import KeyedVectors
import gzip

# Path to the compressed file
file_path_gz = "/content/drive/MyDrive/ML_data/GoogleNews-vectors-negative300.bin.gz"

# Path to save the extracted bin file
file_path_bin = "/content/drive/MyDrive/ML_data/GoogleNews-vectors-negative300.bin"

# Extract the compressed gzip file
with gzip.open(file_path_gz, 'rb') as f_in, open(file_path_bin, 'wb') as f_out:
    f_out.writelines(f_in)


In [ ]:
# Load the Word2Vec model
w2v = KeyedVectors.load_word2vec_format(file_path_bin, binary=True)
print("Word2Vec model loaded successfully.")


Word2Vec model loaded successfully.


In [ ]:
print("Word2Vec 모델의 어휘 크기:", len(w2v.key_to_index))
print("Word2Vec 모델의 어휘:", list(w2v.key_to_index.keys()))

Word2Vec 모델의 어휘 크기: 3000000
Word2Vec 모델의 어휘: 

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [ ]:
All_tags = set()
for tags in eat_info['tags']:
    All_tags.update(set(tags.split(',')))


In [ ]:
new_df = pd.DataFrame()
for word in All_tags:
    try:
        f_vec = w2v.get_vector(word)
        new_df[word] = f_vec
    except:
        pass

In [ ]:
# new_df 출력
print(new_df)

          sub  unreasonable   sausage      meat      tang   foreign  \
0   -0.208984      0.051758 -0.183594 -0.253906 -0.144531 -0.201172   
1    0.223633      0.036133 -0.166992  0.137695  0.000797  0.114258   
2    0.102051      0.060303  0.052002  0.084473  0.053223  0.114258   
3    0.059814      0.063965  0.414062  0.221680 -0.202148  0.050537   
4   -0.164062     -0.166016 -0.134766 -0.012695 -0.020752 -0.068848   
..        ...           ...       ...       ...       ...       ...   
295 -0.011230      0.081055 -0.261719  0.048584 -0.199219  0.050293   
296 -0.171875     -0.008606  0.184570  0.165039 -0.050293  0.096191   
297 -0.008911      0.159180  0.128906  0.075684 -0.148438  0.046875   
298 -0.077148     -0.147461 -0.055420  0.170898 -0.067871  0.112793   
299  0.094238     -0.067383 -0.012878  0.055420  0.345703  0.287109   

     leisurely   musical  overwhelming      bruk  ...  residential     loose  \
0     0.014465  0.110840      0.082520  0.024536  ...    -0.094727 

In [ ]:
x = new_df.T
cluster_count=120
model = KMeans(n_clusters=cluster_count)
model.fit(x)
model.predict(x)
Y=x.copy()
Y['kmeans_id'] = model.predict(x)

/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [ ]:
km = list(Y['kmeans_id'])
max_count = 0
max_count_id = ''
for i in range (0, cluster_count):
    if km.count(i) > max_count:
        max_count = km.count(i);
        max_count_id = i;
    print(i,': ', km.count(i))
max_count_id

0 :  19
1 :  46
2 :  14
3 :  2
4 :  2
5 :  5
6 :  27
7 :  40
8 :  7
9 :  25
10 :  6
11 :  37
12 :  21
13 :  2
14 :  20
15 :  33
16 :  4
17 :  10
18 :  15
19 :  3
20 :  1
21 :  5
22 :  8
23 :  16
24 :  157
25 :  1
26 :  8
27 :  34
28 :  24
29 :  13
30 :  6
31 :  16
32 :  26
33 :  26
34 :  5
35 :  58
36 :  20
37 :  1
38 :  12
39 :  16
40 :  1
41 :  9
42 :  7
43 :  3
44 :  5
45 :  1
46 :  6
47 :  2
48 :  6
49 :  11
50 :  30
51 :  4
52 :  4
53 :  1
54 :  16
55 :  5
56 :  10
57 :  2
58 :  11
59 :  2
60 :  20
61 :  5
62 :  8
63 :  13
64 :  22
65 :  1
66 :  24
67 :  21
68 :  45
69 :  2
70 :  5
71 :  1
72 :  10
73 :  1
74 :  79
75 :  9
76 :  22
77 :  10
78 :  2
79 :  1
80 :  43
81 :  1
82 :  103
83 :  6
84 :  2
85 :  1
86 :  16
87 :  4
88 :  7
89 :  31
90 :  24
91 :  1
92 :  38
93 :  127
94 :  1
95 :  4
96 :  1
97 :  5
98 :  13
99 :  2
100 :  1
101 :  3
102 :  2
103 :  38
104 :  1
105 :  1
106 :  13
107 :  3
108 :  7
109 :  26
110 :  7
111 :  31
112 :  2
113 :  9
114 :  20
115 :  11
116 :  8
1

24

In [ ]:
km_df = pd.DataFrame()
km_df['group_'+str(max_count_id)] = pd.Series(list(Y[Y['kmeans_id'] == max_count_id].index))
for i in range(0, cluster_count):
    col_name = 'group_'+str(i)
    words = list(Y[Y['kmeans_id'] == i].index)
    km_df[col_name] = pd.Series(words)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
km_df.fillna(0)

<ipython-input-30-30d2f3fd091d>:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  km_df[col_name] = pd.Series(words)
<ipython-input-30-30d2f3fd091d>:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  km_df[col_name] = pd.Series(words)
<ipython-input-30-30d2f3fd091d>:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fram

,group_24,group_0,group_1,group_2,group_3,group_4,group_5,group_6,group_7,group_8,group_9,group_10,group_11,group_12,group_13,group_14,group_15,group_16,group_17,group_18,group_19,group_20,group_21,group_22,group_23,group_25,group_26,group_27,group_28,group_29,group_30,group_31,group_32,group_33,group_34,group_35,group_36,group_37,group_38,group_39,group_40,group_41,group_42,group_43,group_44,group_45,group_46,group_47,group_48,group_49,group_50,group_51,group_52,group_53,group_54,group_55,group_56,group_57,group_58,group_59,group_60,group_61,group_62,group_63,group_64,group_65,group_66,group_67,group_68,group_69,group_70,group_71,group_72,group_73,group_74,group_75,group_76,group_77,group_78,group_79,group_80,group_81,group_82,group_83,group_84,group_85,group_86,group_87,group_88,group_89,group_90,group_91,group_92,group_93,group_94,group_95,group_96,group_97,group_98,group_99,group_100,group_101,group_102,group_103,group_104,group_105,group_106,group_107,group_108,group_109,group_110,group_111,group_112,group_113,group_114,group_115,group_116,group_117,group_118,group_119
0,sub,messy,ignorant,flounder,mammoth,boundary,clogged,slim,overwhelming,dazzling,sluggish,sudden,packed,banana,high,bossy,sausage,playful,ok,musical,intuitive,tactile,flavorful,naturalistic,british,patty,concerned,talked,scary,humble,mascarpone,risky,tang,commonplace,hygienic,tingling,snowy,breathable,kitchen,historical,alli,walnut,aroma,nutrition,culinary,mall,ribs,abundant,fish,day,infinite,ale,protein,extinct,excited,iced,syrup,leek,magnetic,remodeled,convenient,meat,hydraulic,ginger,unnamed,servings,strong,thorough,bruk,bilateral,pulmonary,diced,feast,stainless,thirst,pearl,spacious,drank,handwritten,middleweight,spoonful,incense,foreign,physical,burnt,odor,discounted,sashimi,cuttlefish,moist,unreasonable,mart,leisurely,arethe,concise,tacos,apologized,lipstick,electric,diet,dims,sheep,refilled,elasticity,burial,suicide,tea,kindergarten,herb,chopped,noodle,purple,spreading,emu,manga,clunky,eager,mandible,sirloin,compatible
1,squeeze,difficult,forgive,mussel,giant,line,tangle,generous,definite,tidy,annoying,uncommon,hidden,peanut,low,macho,chili,gentle,i,comic,smart,0,chewy,secluded,chinese,0,afraid,know,surprising,passionate,balsamic,rotten,scones,likely,sanitary,nail,sunny,0,tavern,reminiscent,0,wooden,olfactory,digestive,cooking,0,bone,plentiful,ahi,lunchtime,passive,wine,fat,0,lucky,snow,licorice,leeks,pixel,renovated,durable,lamb,adjustable,garlic,visited,0,nucleus,adhered,thai,trilateral,abdominal,0,brunch,0,unpleasantness,sari,classy,eats,handwriting,0,cone,0,unmanned,cognitive,torched,0,upgrade,vegan,eel,acidic,absurd,0,crowded,deeper,0,nachos,0,skin,nitrogen,dietary,0,cow,refill,variable,0,0,wheat,pupil,ginkgo,dried,dumplings,fluorescent,spread,horse,teppanyaki,cluttered,willing,0,steaks,fit
2,normal,delicate,frustrated,salmon,0,0,tangled,diversified,serious,neat,sore,abnormal,accompanied,flowering,0,kicky,spaghetti,loving,hello,art,intelligent,0,crispy,sandy,spanish,0,anxious,aware,embarrassing,careful,parmesan,unsanitary,roasted,inevitable,hygiene,unsighted,rainy,0,upstairs,modern,0,acorn,tastes,nutritional,baking,0,thumb,0,anchovy,seasonal,accurate,liquor,fatty,0,confident,ice,sherbet,0,dialectical,0,cheaper,beef,rotary,basil,opposed,0,ideal,reasonable,hae,0,intestinal,0,dinner,0,habitual,handmade,luxury,ate,0,0,egg,0,parking,mental,0,0,cut,vegetarian,crayfish,elastic,invalid,0,quiet,elsewhere,0,nacho,0,moisturized,environmental,0,0,cattle,0,preferential,0,0,soybean,elementary,herbal,crushed,noodles,darker,0,cats,ramen,cumbersome,reluctant,0,tenderloin,fits
3,fresh,harsh,bored,shrimp,0,0,paralyzed,large,tremendous,colorful,frustrating,unexpected,delayed,strawberry,0,chubby,pancake,affectionate,eatin,music,0,0,crunchy,lush,french,0,worried,wish,suspicious,empathetic,risotto,dirty,delicacies,essential,neatness,bloody,hot,0,rooftop,classic,0,marble,texture,0,cooked,0,tendon,0,mackerel,daytime,unlimited,whiskey,lactose,0,pleased,skate,avocado,0,a

In [ ]:
# remove groups we don't need
# rm_groups = [17, 26, 50, 63]
# for i in rm_groups:
#     group_name = 'group_'+str(i);
#     stop_words.update(set(km_df[group_name].dropna()))
# with open('../data/stop_words.txt','w',-1,'utf-8') as f:
#     f.write('/'.join(stop_words))
eat_info.set_index(eat_info['p_id'],inplace=True,drop=True)
del eat_info['p_id']
eat_info.head()

,name,address,category,main_mn,price,opng_tm,rating,rvw_cnt,tags,reviews
p_id,,,,,,,,,,
304,놀부만두,서울특별시 동대문구 휘경2동 276-33,NaN,냉모밀,6000.0,"{'월-금,일': '오전 11시 - 오후 8시 30분 '}",NaN,NaN,"local,shabby,small,fullthe,cold,deep,thin,bite...","Like a local restaurant, it was moderately sha..."
2234,호반,서울특별시 종로구 낙원동 85-7,NaN,순대국,8000.0,{'매일': '오전 11시 30분 - 오후 10시 (준비시간 15:00~16:30)...,NaN,NaN,"korean,delicious,greatchan,delicioustelephone,...",It is a Korean restaurant located at the edge ...
770,민벅 건대점,서울특별시 광진구 화양동 10-1,NaN,알리오올리오,9900.0,{'매일': '오전 11시 30분 - 오후 10시 (구정 추석 당일 3시 오픈)'},NaN,NaN,"fresh,texture,high,common,cheese,frozen,delici...","If you have 1 menu per person, it will give yo..."
311,뉴델리,서울시 동대문구 회기동 3-182,NaN,치킨 탄두리 (한마리),15000.0,{'매일': '오전 11시 - 오후 10시 '},NaN,NaN,"delicious,indian,old,indian,systemic,long,exot...",You can enjoy delicious Indian curry at an old...
81,강릉스낵,서울특별시 양천구 목동 406-65,NaN,직송 모듬 회,60000.0,"{'월-금': '오후 5시 - 오전 1시 ', '토/일': '오후 4시 - 오전 1...",NaN,NaN,"deliciousthe,fried,crispy,light,delicious,ligh...","Tteokbokki is spicy, so it's delicious.The fri..."


In [ ]:
res_matrix = pd.DataFrame(index=eat_info.index,columns=list(km_df.columns))
user_matrix = pd.DataFrame(index=eat_review_by_user.index,columns=list(km_df.columns))
res_matrix = res_matrix.fillna(0)
user_matrix = user_matrix.fillna(0)

In [ ]:
res_matrix.head()

,group_24,group_0,group_1,group_2,group_3,group_4,group_5,group_6,group_7,group_8,group_9,group_10,group_11,group_12,group_13,group_14,group_15,group_16,group_17,group_18,group_19,group_20,group_21,group_22,group_23,group_25,group_26,group_27,group_28,group_29,group_30,group_31,group_32,group_33,group_34,group_35,group_36,group_37,group_38,group_39,group_40,group_41,group_42,group_43,group_44,group_45,group_46,group_47,group_48,group_49,group_50,group_51,group_52,group_53,group_54,group_55,group_56,group_57,group_58,group_59,group_60,group_61,group_62,group_63,group_64,group_65,group_66,group_67,group_68,group_69,group_70,group_71,group_72,group_73,group_74,group_75,group_76,group_77,group_78,group_79,group_80,group_81,group_82,group_83,group_84,group_85,group_86,group_87,group_88,group_89,group_90,group_91,group_92,group_93,group_94,group_95,group_96,group_97,group_98,group_99,group_100,group_101,group_102,group_103,group_104,group_105,group_106,group_107,group_108,group_109,group_110,group_111,group_112,group_113,group_114,group_115,group_116,group_117,group_118,group_119
p_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
304,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2234,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
770,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
311,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
81,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
user_matrix.head()

,group_24,group_0,group_1,group_2,group_3,group_4,group_5,group_6,group_7,group_8,group_9,group_10,group_11,group_12,group_13,group_14,group_15,group_16,group_17,group_18,group_19,group_20,group_21,group_22,group_23,group_25,group_26,group_27,group_28,group_29,group_30,group_31,group_32,group_33,group_34,group_35,group_36,group_37,group_38,group_39,group_40,group_41,group_42,group_43,group_44,group_45,group_46,group_47,group_48,group_49,group_50,group_51,group_52,group_53,group_54,group_55,group_56,group_57,group_58,group_59,group_60,group_61,group_62,group_63,group_64,group_65,group_66,group_67,group_68,group_69,group_70,group_71,group_72,group_73,group_74,group_75,group_76,group_77,group_78,group_79,group_80,group_81,group_82,group_83,group_84,group_85,group_86,group_87,group_88,group_89,group_90,group_91,group_92,group_93,group_94,group_95,group_96,group_97,group_98,group_99,group_100,group_101,group_102,group_103,group_104,group_105,group_106,group_107,group_108,group_109,group_110,group_111,group_112,group_113,group_114,group_115,group_116,group_117,group_118,group_119
1616,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
905,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1817,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3911,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2151,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
for id in tqdm(res_matrix.index):
    tags = Counter(eat_info.loc[id,'tags'].split(','))
    for tag in tags.keys():
        for num in range(0,cluster_count):
            if tag in set(km_df['group_'+str(num)]):
                res_matrix.loc[id,'group_'+str(num)] += tags[tag]

  0%|          | 0/2314 [00:00<?, ?it/s]

In [ ]:
res_matrix.head()

,group_24,group_0,group_1,group_2,group_3,group_4,group_5,group_6,group_7,group_8,group_9,group_10,group_11,group_12,group_13,group_14,group_15,group_16,group_17,group_18,group_19,group_20,group_21,group_22,group_23,group_25,group_26,group_27,group_28,group_29,group_30,group_31,group_32,group_33,group_34,group_35,group_36,group_37,group_38,group_39,group_40,group_41,group_42,group_43,group_44,group_45,group_46,group_47,group_48,group_49,group_50,group_51,group_52,group_53,group_54,group_55,group_56,group_57,group_58,group_59,group_60,group_61,group_62,group_63,group_64,group_65,group_66,group_67,group_68,group_69,group_70,group_71,group_72,group_73,group_74,group_75,group_76,group_77,group_78,group_79,group_80,group_81,group_82,group_83,group_84,group_85,group_86,group_87,group_88,group_89,group_90,group_91,group_92,group_93,group_94,group_95,group_96,group_97,group_98,group_99,group_100,group_101,group_102,group_103,group_104,group_105,group_106,group_107,group_108,group_109,group_110,group_111,group_112,group_113,group_114,group_115,group_116,group_117,group_118,group_119
p_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
304,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2234,1,1,0,0,0,0,0,3,0,0,0,0,1,0,0,0,3,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,5,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,3,0,0,0,0,0,2,0,2,0,0,0,0,0,0,3,0,0,1,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
770,4,0,0,0,0,0,0,2,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,3,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
311,3,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
81,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0


In [ ]:
for id in tqdm(user_matrix.index):
    tags = Counter(eat_review_by_user.loc[id,'tags'].split(','))
    for tag in tags.keys():
        for num in range(0,cluster_count):
            if tag in set(km_df['group_'+str(num)]):
                user_matrix.loc[id,'group_'+str(num)] += tags[tag]

  0%|          | 0/6181 [00:00<?, ?it/s]

In [ ]:
user_matrix.head()

,group_24,group_0,group_1,group_2,group_3,group_4,group_5,group_6,group_7,group_8,group_9,group_10,group_11,group_12,group_13,group_14,group_15,group_16,group_17,group_18,group_19,group_20,group_21,group_22,group_23,group_25,group_26,group_27,group_28,group_29,group_30,group_31,group_32,group_33,group_34,group_35,group_36,group_37,group_38,group_39,group_40,group_41,group_42,group_43,group_44,group_45,group_46,group_47,group_48,group_49,group_50,group_51,group_52,group_53,group_54,group_55,group_56,group_57,group_58,group_59,group_60,group_61,group_62,group_63,group_64,group_65,group_66,group_67,group_68,group_69,group_70,group_71,group_72,group_73,group_74,group_75,group_76,group_77,group_78,group_79,group_80,group_81,group_82,group_83,group_84,group_85,group_86,group_87,group_88,group_89,group_90,group_91,group_92,group_93,group_94,group_95,group_96,group_97,group_98,group_99,group_100,group_101,group_102,group_103,group_104,group_105,group_106,group_107,group_108,group_109,group_110,group_111,group_112,group_113,group_114,group_115,group_116,group_117,group_118,group_119
1616,5,0,2,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,5,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,4,0,0,0,0,0,0,4,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0
905,18,4,1,0,0,0,0,4,1,0,0,2,0,0,0,0,1,0,0,0,0,0,2,0,2,0,0,0,0,0,0,0,20,0,0,0,24,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,4,0,1,0,0,0,2,0,0,2,1,0,3,2,0,0,0,0,1,0,3,0,3,0,0,0,0,0,4,0,0,0,0,2,0,5,0,0,1,9,0,0,0,0,1,0,0,0,0,3,0,0,0,0,0,0,5,3,0,0,1,5,3,0,0,0
1817,11,5,3,1,0,0,0,23,4,0,12,0,1,0,1,0,0,0,0,0,0,0,3,0,0,0,0,2,2,0,0,4,68,0,0,0,9,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,3,0,0,0,0,0,0,1,0,5,0,0,4,2,0,0,0,0,0,0,5,0,1,0,0,0,1,0,3,0,0,0,0,1,1,15,1,0,4,20,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,3,7,0,0,1,12,1,0,0,1
3911,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,2,0,0,0,0,2,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2151,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
user_matrix.head()

,group_24,group_0,group_1,group_2,group_3,group_4,group_5,group_6,group_7,group_8,group_9,group_10,group_11,group_12,group_13,group_14,group_15,group_16,group_17,group_18,group_19,group_20,group_21,group_22,group_23,group_25,group_26,group_27,group_28,group_29,group_30,group_31,group_32,group_33,group_34,group_35,group_36,group_37,group_38,group_39,group_40,group_41,group_42,group_43,group_44,group_45,group_46,group_47,group_48,group_49,group_50,group_51,group_52,group_53,group_54,group_55,group_56,group_57,group_58,group_59,group_60,group_61,group_62,group_63,group_64,group_65,group_66,group_67,group_68,group_69,group_70,group_71,group_72,group_73,group_74,group_75,group_76,group_77,group_78,group_79,group_80,group_81,group_82,group_83,group_84,group_85,group_86,group_87,group_88,group_89,group_90,group_91,group_92,group_93,group_94,group_95,group_96,group_97,group_98,group_99,group_100,group_101,group_102,group_103,group_104,group_105,group_106,group_107,group_108,group_109,group_110,group_111,group_112,group_113,group_114,group_115,group_116,group_117,group_118,group_119
1616,5,0,2,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,5,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,4,0,0,0,0,0,0,4,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0
905,18,4,1,0,0,0,0,4,1,0,0,2,0,0,0,0,1,0,0,0,0,0,2,0,2,0,0,0,0,0,0,0,20,0,0,0,24,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,4,0,1,0,0,0,2,0,0,2,1,0,3,2,0,0,0,0,1,0,3,0,3,0,0,0,0,0,4,0,0,0,0,2,0,5,0,0,1,9,0,0,0,0,1,0,0,0,0,3,0,0,0,0,0,0,5,3,0,0,1,5,3,0,0,0
1817,11,5,3,1,0,0,0,23,4,0,12,0,1,0,1,0,0,0,0,0,0,0,3,0,0,0,0,2,2,0,0,4,68,0,0,0,9,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,3,0,0,0,0,0,0,1,0,5,0,0,4,2,0,0,0,0,0,0,5,0,1,0,0,0,1,0,3,0,0,0,0,1,1,15,1,0,4,20,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,3,7,0,0,1,12,1,0,0,1
3911,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,2,0,0,0,0,2,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2151,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
km_df.count()

group_24     157
group_0       19
group_1       46
group_2       14
group_3        2
group_4        2
group_5        5
group_6       27
group_7       40
group_8        7
group_9       25
group_10       6
group_11      37
group_12      21
group_13       2
group_14      20
group_15      33
group_16       4
group_17      10
group_18      15
group_19       3
group_20       1
group_21       5
group_22       8
group_23      16
group_25       1
group_26       8
group_27      34
group_28      24
group_29      13
group_30       6
group_31      16
group_32      26
group_33      26
group_34       5
group_35      58
group_36      20
group_37       1
group_38      12
group_39      16
group_40       1
group_41       9
group_42       7
group_43       3
group_44       5
group_45       1
group_46       6
group_47       2
group_48       6
group_49      11
group_50      30
group_51       4
group_52       4
group_53       1
group_54      16
group_55       5
group_56      10
group_57       2
group_58      

In [ ]:
eat_review_score = pd.DataFrame(eat_review, columns=['u_id','p_id','rating'])
eat_review_score.sort_values(by=['u_id'], axis=0,inplace=True)
eat_review_score.reset_index(drop = True, inplace=True)
eat_review_score.head()

,u_id,p_id,rating
0,0,720,4
1,0,360,4
2,1,1656,5
3,1,749,5
4,2,2032,3


In [ ]:
res_matrix.to_csv('/content/drive/MyDrive/ML_data/res_matrix.csv')
user_matrix.to_csv('/content/drive/MyDrive/ML_data/user_matrix.csv')

eat_review_score.to_csv('/content/drive/MyDrive/ML_data/score_board.csv',index=False)

# 클러스터링 결과
km_df.to_csv('/content/drive/MyDrive/ML_data/eat_cluster.csv',index=False)

km_df.count().to_csv('/content/drive/MyDrive/ML_data/group_count.csv')

NCF 모델

In [ ]:
!pip install cornac

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 38.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import cornac
import cornac.eval_methods
from cornac.eval_methods import RatioSplit, base_method
from cornac.hyperopt import Continuous, Discrete, RandomSearch
from tqdm import tqdm

import warnings
warnings.filterwarnings(action='ignore')
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

In [ ]:
# 추천할 top k개의 장소
TOP_K = 10
data_df = pd.read_csv('/content/drive/MyDrive/ML_data/score_board (1).csv', index_col =0)
data_df['rating'] =data_df['rating'].astype('float')

In [ ]:
# # 'u_id'를 열로 변환
# data_df.reset_index(inplace=True)

# 데이터가 잘 처리되었는지 확인
print(data_df)

       u_id  p_id  rating
0         0  3375     3.0
1         0  2778     5.0
2         0   612     5.0
3         0  3949     5.0
4         0  1679     1.0
...     ...   ...     ...
105198    쥬  1679     NaN
105199    쥬  3848     NaN
105200    쥬  1547     NaN
105201    쥬  4080     NaN
105202    쥬  3754     NaN

[105203 rows x 3 columns]


In [ ]:
data_df = data_df.rename({'u_id': 'userID', 'p_id': 'itemID'}, axis=1)

In [ ]:
print(data_df.head())

  userID  itemID  rating
0      0    3375     3.0
1      0    2778     5.0
2      0     612     5.0
3      0    3949     5.0
4      0    1679     1.0


In [ ]:
data = [tuple(x) for x in data_df.values]
data

[('0', 3375, 3.0),
 ('0', 2778, 5.0),
 ('0', 612, 5.0),
 ('0', 3949, 5.0),
 ('0', 1679, 1.0),
 ('0', 4805, 5.0),
 ('0', 287, 3.0),
 ('0', 3601, 5.0),
 ('0', 3526, 5.0),
 ('0', 3601, 5.0),
 ('0', 3357, 5.0),
 ('0', 104, 5.0),
 ('0', 2307, 5.0),
 ('0', 4345, 5.0),
 ('0', 3075, 5.0),
 ('0', 1269, 5.0),
 ('0', 1714, 5.0),
 ('0', 479, 5.0),
 ('0', 1186, 5.0),
 ('0', 3388, 5.0),
 ('0', 5068, 5.0),
 ('0', 5264, 3.0),
 ('0', 1123, 5.0),
 ('0', 3734, 5.0),
 ('0', 1904, 3.0),
 ('0', 4843, 5.0),
 ('0', 616, 5.0),
 ('0', 1092, 5.0),
 ('0', 3529, 5.0),
 ('0', 1254, 5.0),
 ('0', 3118, 5.0),
 ('1', 3612, 5.0),
 ('2', 3441, 5.0),
 ('2', 1737, 5.0),
 ('3', 5334, 5.0),
 ('4', 1127, 5.0),
 ('4', 1846, 5.0),
 ('4', 3415, 5.0),
 ('4', 1666, 5.0),
 ('4', 208, 5.0),
 ('4', 1846, 5.0),
 ('4', 709, 3.0),
 ('4', 177, 5.0),
 ('4', 3415, 5.0),
 ('4', 1846, 5.0),
 ('4', 1368, 5.0),
 ('4', 2382, 5.0),
 ('4', 5153, 5.0),
 ('4', 3761, 5.0),
 ('4', 3761, 5.0),
 ('4', 2910, 5.0),
 ('4', 2091, 5.0),
 ('4', 177, 5.0),
 (

In [ ]:
# Instantiate evaluation metrics
ndcg = cornac.metrics.NDCG(k=TOP_K)
pre = cornac.metrics.Precision(k=TOP_K)
rec = cornac.metrics.Recall(k=TOP_K)
fm = cornac.metrics.FMeasure(k=TOP_K)

In [ ]:
# Assuming data is a list of records
for record in data:
    print(record)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
('21159', 4599, 1.0)
('21159', 5337, 3.0)
('21159', 1811, 5.0)
('21159', 4680, 3.0)
('21160', 920, 5.0)
('21160', 4899, 5.0)
('21160', 1618, 5.0)
('21160', 4080, 5.0)
('21160', 5273, 5.0)
('21161', 1853, 4.0)
('21162', 2632, 5.0)
('21162', 2278, 5.0)
('21162', 3931, 5.0)
('21162', 4142, 5.0)
('21163', 3673, 5.0)
('21164', 2309, 1.0)
('21164', 585, 1.0)
('21165', 529, 5.0)
('21165', 1125, 3.0)
('21165', 412, 5.0)
('21165', 4558, 5.0)
('21165', 1789, 5.0)
('21165', 4576, 1.0)
('21165', 538, 5.0)
('21165', 5011, 1.0)
('21165', 4153, 5.0)
('21165', 4694, 5.0)
('21165', 4274, 1.0)
('21165', 1646, 5.0)
('21165', 3233, 5.0)
('21165', 3218, 5.0)
('21165', 4539, 5.0)
('21165', 948, 5.0)
('21165', 2278, 5.0)
('21165', 4224, 5.0)
('21165', 2283, 5.0)
('21165', 2864, 5.0)
('21165', 1860, 3.0)
('21165', 948, 5.0)
('21165', 920, 5.0)
('21165', 4734, 5.0)
('21165', 4734, 5.0)
('21165', 4525, 3.0)
('21165', 1366, 5.0)
('21165', 4734, 5.0)
('21165', 3450, 3.0)
('2116

In [ ]:
#Define an evaluation method to split feedback into train and test sets
ratio_split = RatioSplit(
    data=data,
    test_size=0.1,
    val_size=0.1,
    rating_threshold=1.0,
    seed=123,
    verbose=True,
)

neumf = cornac.models.NeuMF(
    layers=[64, 32, 16, 8],
    act_fn="tanh",
    learner="adam",
    num_neg=50,
    seed=123,
    early_stopping = {'min_delta':0.0, 'patience':5}
)

# RandomSearch
rs_neumf = RandomSearch(
    model = neumf,
    space=[
        Discrete("num_epochs", [50, 100, 150, 200]),
        Discrete("num_factors", [4, 8]),
        Discrete("batch_size", [128, 256, 512]),
        Continuous("lr", 0.001, 0.01)
    ],
    metric = fm,
    eval_method = ratio_split
)

# Put everything together into an experiment and run it
cornac.Experiment(
    eval_method=ratio_split,
    models=[rs_neumf],
    metrics=[ndcg, pre, rec, fm],
).run()


rating_threshold = 1.0
exclude_unknowns = True
---
Training data:
Number of users = 19705
Number of items = 5318
Number of ratings = 81784
Max rating = nan
Min rating = nan
Global mean = nan
---
Test data:
Number of users = 4275
Number of items = 2920
Number of ratings = 9116
Number of unknown users = 0
Number of unknown items = 0
---
Validation data:
Number of users = 4285
Number of items = 2990
Number of ratings = 9114
---
Total users = 19705
Total items = 5318

[RandomSearch_NeuMF] Training started!
Evaluating: {'batch_size': 512, 'lr': 0.007416597884709045, 'num_epochs': 150, 'num_factors': 4}


  0%|          | 0/150 [00:00<?, ?it/s]

Early stopping:
- best epoch = 19, stopped epoch = 24
- best monitored value = 0.077429 (delta = -0.001048)
Evaluating: {'batch_size': 512, 'lr': 0.005961832921746022, 'num_epochs': 200, 'num_factors': 4}


  0%|          | 0/200 [00:00<?, ?it/s]

Early stopping:
- best epoch = 37, stopped epoch = 42
- best monitored value = 0.078229 (delta = -0.002393)
Evaluating: {'batch_size': 256, 'lr': 0.00982687778546154, 'num_epochs': 50, 'num_factors': 8}


  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
print('Random search: ', rs_neumf.best_params)

In [ ]:
ratio_split = RatioSplit(
    data=data,
    test_size=0,
    rating_threshold=1.0,
    verbose=True,
)

neumf = cornac.models.NeuMF(
    num_factors = rs_neumf.best_params['num_factors'],
    layers=[64, 32, 16, 8],
    act_fn="tanh",
    learner="adam",
    num_epochs = rs_neumf.best_params['num_epochs'],
    lr=rs_neumf.best_params['lr'],
    num_neg=50,
    batch_size = rs_neumf.best_params['batch_size'],
    seed=123,
    early_stopping = {'min_delta':0.0, 'patience':5}
)

cornac.Experiment(
    eval_method=ratio_split,
    models=[neumf],
    metrics=[ndcg, pre, rec, fm],
).run()

In [ ]:
import pandas as pd
pred_user = pd.read_csv('/content/drive/MyDrive/ML_data/survey_result.csv')
pred_user = set(pred_user['u_id'])

In [ ]:
print(data_df.head())

In [ ]:
all_prediction = pd.DataFrame(columns=['u_id','p_id','ncf_score'])

for u_idx, u_id in tqdm(enumerate(data_df['userID'].unique())):
    if u_id in pred_user:
        tmp = pd.DataFrame(neumf.rank(u_idx)).T
        tmp['u_id'] = u_id
        tmp = pd.DataFrame(tmp,columns=['u_id',0,1])
        tmp.rename(columns={0:'p_id',1:'ncf_score'}, inplace=True)
        all_prediction = all_prediction.append(tmp)

In [ ]:
all_prediction

In [ ]:
all_prediction.to_csv('/content/drive/MyDrive/ML_data/predictions/ncf_scores.scv',index=False)

CBF 모델

In [ ]:
import pandas as pd
import numpy as np
import math
# get size of vector
def mag(x):
    return math.sqrt(sum(i**2 for i in x))
res_matrix = pd.read_csv('/content/drive/MyDrive/ML_data/res_matrix.csv')
user_matrix = pd.read_csv('/content/drive/MyDrive/ML_data/user_adj_matrix.csv')
group_count = pd.read_csv('/content/drive/MyDrive/ML_data/group_count.csv',names=['group','count'])

res_matrix.set_index('p_id', drop=True, inplace=True)
user_matrix.set_index('u_id', drop=True, inplace=True)
group_count.set_index('group', drop=True, inplace=True)
user_matrix

최종